# AC-Zero — Scheduler Runner (Kaggle)

The **single** notebook the GitHub Actions scheduler launches for every task. It
reads a secret-free `runtime_config.json` the scheduler dropped next to it,
branches on `mode` (`generation` / `annotation` / `training`), reports
heartbeats/status to the private Hugging Face state repo, and stops cleanly at a
soft runtime deadline (well before Kaggle's ~12 h hard kill).

**Do not edit the config cells by hand** — the scheduler owns them. Manual
control happens by editing the HF `queue.yaml` or via the workflow dispatch.

The Hugging Face token arrives through the private Kaggle **runtime-secrets**
dataset (`/kaggle/input/runtime-secrets/hf_token.txt`); it is never printed,
saved to `/kaggle/working`, or written into any output.

## 1. Install AC-Zero from GitHub

In [ ]:
import subprocess
import sys

REPO_URL = "https://github.com/HkHk2Prod/AlphaAC.git"
REPO_BRANCH = "main"

def pip_install(*args):
    proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--progress-bar", "off", *args],
        capture_output=True, text=True,
    )
    if proc.returncode:
        print(proc.stdout, proc.stderr, sep="\n"); proc.check_returncode()

pip_install("--no-deps", f"git+{REPO_URL}@{REPO_BRANCH}")
pip_install("numpy>=1.26", "pydantic>=2", "pyyaml>=6", "typer>=0.12", "rich>=13",
            "gymnasium>=0.29", "matplotlib>=3.8", "huggingface_hub>=1.21")
print("Installed ac_zero from", REPO_BRANCH)

## 2. Runtime config, Hugging Face login, run reporter

In [ ]:
import os

from ac_zero.scheduler.notebook import RunReporter, load_runtime_config, login_from_secret_dataset

CFG = load_runtime_config("runtime_config.json")
TASK_ID = CFG["task_id"]; RUN_ID = CFG["run_id"]; MODE = CFG["mode"]
MAX_RUNTIME_MIN = int(CFG.get("max_runtime_minutes", 705))
STATE_REPO = CFG["hf_state_repo_id"]; STATE_REPO_TYPE = CFG.get("hf_state_repo_type", "dataset")
OPTS = CFG.get("config", {})
print(f"task={TASK_ID} run={RUN_ID} mode={MODE} max_runtime_min={MAX_RUNTIME_MIN}")

# Read the HF token from the private Kaggle dataset and log in (never printed).
_hf_token = login_from_secret_dataset()
os.environ["HF_TOKEN"] = _hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = _hf_token

reporter = RunReporter(STATE_REPO, run_id=RUN_ID, task_id=TASK_ID,
                       token=_hf_token, repo_type=STATE_REPO_TYPE)
reporter.started(extra={"accelerator": CFG.get("accelerator")})
print("reported run start to", STATE_REPO)

## 3. Soft deadline + heartbeat / stop-signal thread

In [ ]:
import threading
import time

START = time.monotonic()
SAFETY_MARGIN_MIN = 10
DEADLINE = START + max(1, MAX_RUNTIME_MIN - SAFETY_MARGIN_MIN) * 60
HEARTBEAT_EVERY_S = 300  # 5 min

# How long a job may run before the watchdog terminates it. A job that is terminated
# is killed outright -- nothing in ac_zero handles SIGTERM -- so whatever it had not
# yet written and pushed is gone. Both long jobs therefore reserve head-room on top of
# the budget and stop themselves early, rather than being cut off mid-flush:
#
#   training  after its last iteration it still writes plots and the certificate and
#             pushes the checkpoint bundle to Hugging Face.
#   ball      after its last batch it still rewrites both documents (the whole ball) and
#             pushes them, which is the only reason the session produced anything at all.
#
# The margins are measured from the *subprocess*, whose clock starts later than the
# watchdog's (the bucket pull happens in between), so each job's own soft budget expires
# comfortably before the watchdog reaches for it.
JOB_BUDGET_MIN = max(1, MAX_RUNTIME_MIN - SAFETY_MARGIN_MIN)
TRAIN_FLUSH_MARGIN_MIN = 10
BALL_FLUSH_MARGIN_MIN = 20

def budget_min(flush_margin_min):
    """Minutes the job may run: what is left of the budget *now*, less its flush margin.

    Measured from here rather than from the session start, because a job is launched only
    after its dataset has been pulled from the bucket -- and for a multi-gigabyte ball that
    is not a rounding error. Budgeting from the session start would hand the job minutes
    that were already spent on the pull, and the overrun would come out of the one place
    it must not: the flush margin, the minutes the job needs to write its output and push
    it before the watchdog reaches for it. Counted from now, the pull comes out of the
    job's own working time and the margin survives intact.
    """
    remaining = (DEADLINE - time.monotonic()) / 60
    return max(1, int(remaining - flush_margin_min))

stop_event = threading.Event()          # set on deadline OR operator stop request
deadline_hit = threading.Event()

def _watchdog():
    while not stop_event.is_set():
        if time.monotonic() >= DEADLINE:
            print("[watchdog] soft deadline reached; requesting stop.")
            deadline_hit.set(); stop_event.set(); break
        try:
            reporter.heartbeat(extra={"elapsed_min": round((time.monotonic()-START)/60, 1)})
            if reporter.should_stop():
                print("[watchdog] operator stop requested via queue; requesting stop.")
                stop_event.set(); break
        except Exception as exc:  # never let a transient HF blip kill the run
            print("[watchdog] heartbeat/stop-check skipped:", exc)
        stop_event.wait(HEARTBEAT_EVERY_S)

_hb = threading.Thread(target=_watchdog, daemon=True); _hb.start()
print(f"watchdog running: soft deadline in {MAX_RUNTIME_MIN - SAFETY_MARGIN_MIN} min, "
      f"heartbeat every {HEARTBEAT_EVERY_S//60} min.")

## 4. Prepare inputs and build the mode command

The same notebook serves every task; `mode` and `config` from the runtime config
select what runs. Each mode maps to an `aczero` CLI subcommand. Annotation, solve
and training pull the grown dataset (and, for the `navigation` reward, its
`strict-ac` annotations) from the shared HF bucket first; training materializes
`config` into a YAML the CLI reads, so no repo `configs/` files are needed on
Kaggle. Generation resumes the bucket dataset and **writes it back**; annotation
writes its refined annotations back; training warm-starts from the best model
under `model_checkpoints/<name>/` and pushes its own bundle back — so each
scheduled run continues the last.

In [ ]:
from pathlib import Path

import yaml

HF_BUCKET_DEFAULT = "HkHk2Prod/alphaac-data"
# Kaggle caps /kaggle/working at 20 GB but the container scratch /kaggle/tmp holds ~60 GB;
# the ball's groups+annotations pair (and training's instance sidecar) outgrows 20 GB well
# before the target, and the durable copy lives in the HF bucket, so stage the big local
# files under /kaggle/tmp. Kaggle does not pre-create it, so make it rather than probe.
WORK_ROOT = Path("/kaggle/tmp") if Path("/kaggle").is_dir() else Path(".")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR = WORK_ROOT / "data" / "generated"; DATA_DIR.mkdir(parents=True, exist_ok=True)

def report_disk_usage():
    # Kaggle's disk quotas are undocumented and vary by container generation, so print
    # the live free/used/total for each mount before the job runs -- the ball and
    # training both stage large files under /kaggle/tmp and can overflow it.
    import shutil

    for label in ["/kaggle", "/kaggle/working", "/kaggle/tmp", str(WORK_ROOT)]:
        if not Path(label).is_dir():
            continue
        total, used, free = shutil.disk_usage(label)
        print(f"[disk] {label}: total={total/1e9:.1f} GB  used={used/1e9:.1f} GB  free={free/1e9:.1f} GB")

report_disk_usage()

def _opt(name, default=None):
    return OPTS.get(name, default)

def _bucket():
    return (OPTS.get("dataset") or {}).get("bucket") or HF_BUCKET_DEFAULT

def pull(remote_name, local_path):
    from ac_zero.datasets.hub import download_dataset
    return download_dataset(local_path, remote_name=remote_name, bucket=_bucket(), missing_ok=True)

# (local_path, remote_name) pairs pushed back to the bucket after a successful run.
UPLOADS = []

def push_outputs():
    # Upload each (local, remote) in UPLOADS; return the remote names pushed.
    from ac_zero.datasets.hub import upload_dataset
    pushed = []
    for local_path, remote_name in UPLOADS:
        p = Path(local_path)
        if not p.exists():
            print(f"[write-back] skip {remote_name}: {p} not found"); continue
        upload_dataset(p, remote_name=remote_name, bucket=_bucket())
        print(f"[write-back] pushed {p.name} -> {_bucket()}/{remote_name}")
        pushed.append(remote_name)
    return pushed

def relator_bound():
    # The relator bound is one number shared by generation and training: the ball is grown
    # under it, the model's encoder is built to it, and a training run whose capacity
    # disagrees with its dataset's bound is refused. A `ball` task states it as
    # `max_relator_length`, a `training` task as `max_relator_tokens` -- the same number
    # under the name each CLI uses, so both resolve to the same file in the bucket.
    return int(_opt("max_relator_length", _opt("max_relator_tokens", 0)) or 0)

def prepare_and_build(mode):
    from ac_zero.datasets.annotate import annotation_path
    from ac_zero.datasets.ball import ball_groups_path

    rank = int(_opt("rank", 2))
    moveset = str(_opt("moveset", "strict-ac"))
    bound = relator_bound()
    # Two datasets can live in the bucket. `ball_rank{rank}_rel{bound}` is grown
    # closest-first *within that relator bound*, so every distance in it is a proven
    # optimum in the graph a model of that encoder capacity actually moves in, and its
    # shells are complete; `train_rank{rank}` is the older length-first grow, whose
    # distances are searched for afterwards and are only upper bounds. Training reads the
    # ball; a task can name the other with `dataset.stem: train_rank2`. The names come
    # from the library helper, so the file this pulls is the file the CLI writes.
    default_stem = ball_groups_path("", rank, bound).name.removesuffix(".groups.json")
    stem = str((OPTS.get("dataset") or {}).get("stem") or default_stem)
    ds_name = f"{stem}.groups.json"
    ann_name = annotation_path(ds_name, moveset).name
    grown_name = f"train_rank{rank}.groups.json"
    grown_ann = f"train_rank{rank}.{moveset}.annotations.json"

    if mode == "ball":
        # Closest-first generation: walk outward from the trivial group by the inverses
        # of the move set's moves, breadth-first. Resume needs both files -- the groups
        # are the queue (expanded in discovery order) and the annotations are the
        # distances they were discovered at -- and both are pushed back.
        output = str(_opt("output", f"{WORK_ROOT}/{ds_name}"))
        ann_output = str(Path(output).parent / ann_name)
        pull(ds_name, output)
        pull(ann_name, ann_output)
        UPLOADS.append((output, ds_name))
        UPLOADS.append((ann_output, ann_name))
        return ["aczero", "dataset", "ball",
                "--rank", str(rank),
                "--moveset", moveset,
                # Grow the ball under the bound the models train at: a move that would
                # overshoot it is one their environment masks, so this is the graph they
                # actually move in -- and a run whose capacity disagrees is refused.
                "--max-relator-length", str(bound),
                "--target", str(_opt("target", 100_000_000)),
                # Fill the run: expand until the soft deadline, not just the target --
                # stopping early enough to rewrite both documents and push them.
                "--minutes", str(budget_min(BALL_FLUSH_MARGIN_MIN)),
                # Every N hours the run rewrites both documents and pushes them to the
                # bucket. That push is the session's only durable output (the container
                # dies with the session), so the interval is how much expansion an
                # interruption may cost -- not a disk-write cadence.
                "--checkpoint-hours", str(_opt("checkpoint_hours", 4)),
                # The bucket copy is the durable one (pulled on resume), so skip the atomic
                # temp copy: it halves the peak disk a rewrite needs and lets the ball grow
                # past where the temp would overflow /kaggle/tmp.
                "--no-atomic-checkpoint",
                "--workers", str(_opt("workers", 0)),
                "--output", output]

    if mode == "generation":
        # The older length-first grow, over the full universal move set.
        output = str(_opt("output", f"{WORK_ROOT}/{grown_name}"))
        pull(grown_name, output)
        UPLOADS.append((output, grown_name))
        return ["aczero", "dataset", "grow",
                "--rank", str(rank),
                "--target", str(_opt("target", 100_000_000)),
                # Fill the run: grow until the soft deadline, not just the target --
                # stopping early enough to flush what it grew.
                "--minutes", str(budget_min(BALL_FLUSH_MARGIN_MIN)),
                "--max-relator-length", str(_opt("max_relator_length", 0)),
                "--select", str(_opt("select", "smallest")),
                "--seed", str(_opt("seed", 0)),
                "--checkpoint-every", str(_opt("checkpoint_every", 1000)),
                "--workers", str(_opt("workers", 0)),
                "--output", output]

    if mode == "annotation":
        # Only a grown dataset needs annotating; a ball writes its distances itself.
        ds_local = DATA_DIR / grown_name
        if pull(grown_name, ds_local) is None:
            raise RuntimeError(f"annotation needs {grown_name} in bucket {_bucket()}; run generation first.")
        ann_local = DATA_DIR / grown_ann
        pull(grown_ann, ann_local)  # warm start from any existing annotations
        UPLOADS.append((ann_local, grown_ann))  # push refined annotations back
        return ["aczero", "dataset", "annotate",
                "--input", str(ds_local),
                "--moveset", moveset,
                "--max-depth", str(_opt("max_depth", 0)),
                "--workers", str(_opt("workers", 0))]

    if mode == "training":
        ds_local, ann_local = DATA_DIR / ds_name, DATA_DIR / ann_name
        if pull(ds_name, ds_local) is None:
            raise RuntimeError(f"training needs {ds_name} in bucket {_bucket()}; run ball generation first.")
        have_ann = pull(ann_name, ann_local) is not None
        reward = (OPTS.get("training") or {}).get("reward_mode")
        if reward in ("navigation", "potential") and not have_ann:
            raise RuntimeError(f"reward_mode {reward!r} needs {ann_name} in bucket; generate the ball first.")
        cfg = {k: v for k, v in OPTS.items() if k != "seed"}
        ds_cfg = {k: v for k, v in (cfg.get("dataset") or {}).items() if k != "stem"}
        ds_cfg["path"] = str(ds_local)
        if have_ann:
            ds_cfg["annotations"] = str(ann_local)
        cfg["dataset"] = ds_cfg
        cfg_path = Path("/kaggle/working/_train_config.yaml")
        cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
        return ["aczero", "train",
                "--config", str(cfg_path),
                "--seed", str(_opt("seed", 0)),
                # Fill the run: the wall-clock budget, not `training.iterations`,
                # ends it -- at an iteration boundary, with time left to upload.
                "--minutes", str(budget_min(TRAIN_FLUSH_MARGIN_MIN)),
                # Continue this checkpoint lineage on HF: warm-start from the best
                # model already there, then push this run's best model + plots back.
                # The bundle is pushed on this cadence and once more at the end. Like the
                # ball's checkpoint, that push is the only durable output: the container
                # dies with the session, so this is how much training a lost session costs.
                "--upload-every-hours", str(_opt("upload_every_hours", 4)),
                "--download-checkpoint",
                "--upload-checkpoints",
                "--checkpoint-bucket", _bucket()]

    if mode == "solve":
        ds_local = DATA_DIR / ds_name
        if pull(ds_name, ds_local) is None:
            raise RuntimeError(f"solve needs {ds_name} in bucket {_bucket()}; run ball generation first.")
        return ["aczero", "solve", "--presentation", str(ds_local), "--agent", str(_opt("agent", "greedy"))]

    raise ValueError(f"unknown mode: {mode!r}")

COMMAND = prepare_and_build(MODE)
print("running:", " ".join(COMMAND))


## 5. Run the job under the deadline, then report status

In [ ]:

status = "finished"; error = None
proc = subprocess.Popen(COMMAND, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    while True:
        if proc.poll() is not None:
            break
        if stop_event.is_set():
            print("[main] stop requested — terminating job so it can checkpoint & exit.")
            proc.terminate()
            try:
                proc.wait(timeout=120)
            except subprocess.TimeoutExpired:
                proc.kill()
            status = "stopped" if not deadline_hit.is_set() else "finished"
            break
        line = proc.stdout.readline() if proc.stdout else ""
        if line:
            print(line, end="")
        else:
            time.sleep(1)
    if proc.stdout:
        for line in proc.stdout:
            print(line, end="")
    rc = proc.wait()
    if rc != 0 and status == "finished":
        status = "failed"; error = f"job exited with code {rc}"
except Exception as exc:
    status = "failed"; error = repr(exc)
finally:
    stop_event.set()

# Write-back: push the grown dataset / refined annotations to the bucket so the
# next scheduled run continues from them. Only on a usable outcome (finished or a
# clean operator stop); grow/annotate checkpoint atomically, so the file is
# consistent. Training pushes its own model checkpoints via the training pipeline.
wrote_back = []
if status in ("finished", "stopped"):
    try:
        wrote_back = push_outputs()
    except Exception as exc:
        print(f"[write-back] failed: {exc}")

# A failed run still counts as a completed scheduled launch — the scheduler does
# NOT restore remaining_runs. We only report status for visibility. `wrote_back`
# records which bucket artifacts this run pushed, for observability.
reporter.finished(status=status, error=error,
                  extra={"wrote_back": wrote_back} if wrote_back else None)
print(f"run {RUN_ID} reported as {status}" + (f" ({error})" if error else ""))
if status == "failed":
    raise SystemExit(error or "job failed")